# Neural Network Hyperparameter Search

TensorFlow/Keras binary classifier on a public email spam benchmark (4,601 emails, 57 word- and character-frequency features), with a manual grid search over batch size and epochs.

*Origin: University of Cambridge Data Science Career Accelerator coursework (2024-25). Data source anonymised and not included. Cell outputs cleared to protect the source data; results quoted in the README come from the original run.*

In [ ]:
# Start your activity here:

# URL to import data set from GitHub.
url = 'data/emails.csv'

In [ ]:
# 1. Importing the libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
tf.random.set_seed(42)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Input
from tensorflow.keras.callbacks import EarlyStopping
import csv
import os # import the os module to use operating system related functions

In [ ]:
# The data set
df = pd.read_csv(url, header=None)

In [ ]:
df.shape

In [ ]:
# 2. Checking the data set
print(df.head())

In [ ]:
# 3. Define features (X) and target (y)
# Here the the information gets separated (like words in emails)
# from what needs to be predicted (if it's spam or not).
# The last column in the data set indicates if each email is spam or not.

X = df.iloc[:, :-1]  # All columns except the last one
y = df.iloc[:, -1]   # Last column for classification

In [ ]:
# 4. Here the data is divided into three parts:
# Training Set: to teach the model how to classify emails
# Validation Set: For practice test to see how well the  model is learning without using the final check (test set).
# Test Set: This is the final exam where it's checked if the model can classify new emails it hasn't seen before.

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) #  42 is chosen to ensure that the random processes in the code are reproducible.

# Create validation set from training data
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42)

In [ ]:
# 5.1 Standardising the features
# making all the data features (like word counts) have the same scale.

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# 5.2 Defining the model

# model = Sequential([
#     Input(shape=(X_train.shape[1],)), # Using the Input layer to define the input shape
#     # the Input layer as the first layer in the Sequential model
#     Dense(64, activation='relu'),
#     Dense(32, activation='relu'),
#     Dense(1, activation='sigmoid')
#     ])

# Define the model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X.shape[1],)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(16, activation='relu'),
    Dense(16, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Notes: The first hidden layer has 64 neurons with ReLU (Rectified Linear Unit)
# activation.

# The second hidden layer has 32 neurons, also with ReLU activation.
# Using a smaller number of neurons can help the network learn hierarchical
# representations and reduce the risk of overfitting.

# The output layer has a single neuron with sigmoid activation.
# Sigmoid is appropriate for binary classification tasks as it outputs a
# probability between 0 and 1, helping to classify the input as either spam or
# non-spam.

In [ ]:
# Compile the model
model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy']) # loss='binary_crossentropy to measure how far off its predictions are from the true outcomes

In [ ]:
# Train the model
history = model.fit(X_train_scaled, y_train,
                    validation_data=(X_val_scaled, y_val),
                    epochs=10, # 10 epochs to avoid overfitting
                    batch_size=64, # 64 is a reasonable initial choice that balances computational efficiency with model performance
                    verbose=1)

In [ ]:
# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

**The Test Loss increases and Test Accuracy starts dominishing with new retraining and re-evaluation attempts**

In [ ]:
# Create a vector of different batch_sizes=np.array([16, 32, 64]) and loop
# through it, retraining the model each time, and print the performances.
# Use the same model and number of epochs.

batch_sizes=np.array([16, 32, 64])
for batch_size in batch_sizes:
  model = Sequential([
    Dense(64, activation='relu', input_shape=(X.shape[1],)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(16, activation='relu'),
    Dense(16, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
  ])

  model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
  history = model.fit(X_train_scaled, y_train,
                      validation_data=(X_val_scaled, y_val),
                      epochs=10,
                      batch_size=batch_size,
                      verbose=1)
  test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
  print(f"Batch Size: {batch_size}")
  print(f"Test Loss: {test_loss:.4f}")
  print(f"Test Accuracy: {test_accuracy:.4f}")
  print()

In [ ]:
# Create a vector of different epochs=np.array([10, 20, 30]) and loop through
# it, retraining the model each time and using the batch size.
# Jot down which model gave you the highest accuracy.

epochs=np.array([10, 20, 30])
for epoch in epochs:
  model = Sequential([
    Dense(64, activation='relu', input_shape=(X.shape[1],)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(16, activation='relu'),
    Dense(16, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid'),
  ])
  # Compile the model again
  model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])
  # Train the model
  history = model.fit(X_train_scaled, y_train,
                      validation_data=(X_val_scaled, y_val),
                      epochs=epoch,
                      batch_size=64,  # Use this for consistency with batch size experiments
                      verbose=1)
  # Evaluate the model
  test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
  print(f"Epochs: {epoch}")
  print(f"Test Loss: {test_loss:.4f}")
  print(f"Test Accuracy: {test_accuracy:.4f}")
  print()

### **Best hyperparameters calculation using the Grid Search approach**

In [ ]:
# Hyperparameters for grid search
batch_sizes = np.array([16, 32, 64])
epochs = np.array([10, 20, 30])

# Lists to store results from multiple runs
best_accuracies = []
best_epochs_list = []
best_batch_sizes = []

for _ in range(100):  # Run 10 times
    best_accuracy = 0
    best_epochs = 0
    best_batch_size = 0

    model = Sequential([
      Dense(64, activation='relu', input_shape=(X.shape[1],)),
      Dense(32, activation='relu'),
      Dense(16, activation='relu'),
      Dense(16, activation='relu'),
      Dense(16, activation='relu'),
      Dense(16, activation='relu'),
      Dense(1, activation='sigmoid'),
    ])

    # Compile the model again
    model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

    # Early stopping callback to avoid overfitting
    early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    for batch_size in batch_sizes:
        for epoch in epochs:
            # Reset the model weights before training
            model = Sequential([
              Dense(64, activation='relu', input_shape=(X.shape[1],)),
              Dense(32, activation='relu'),
              Dense(16, activation='relu'),
              Dense(16, activation='relu'),
              Dense(16, activation='relu'),
              Dense(16, activation='relu'),
              Dense(1, activation='sigmoid'),
            ])
            model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

            # Train the model
            history = model.fit(X_train_scaled, y_train,
                                validation_data=(X_val_scaled, y_val),
                                epochs=epoch,
                                batch_size=batch_size,
                                callbacks=[early_stopping],
                                verbose=0)  # Setting verbose to 0 to reduce output for 10 runs
            # Evaluate the model
            test_loss, test_accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)

            # Record the best hyperparameters
            if test_accuracy > best_accuracy:
                best_accuracy = test_accuracy
                best_epochs = epoch
                best_batch_size = batch_size

    # Store the best parameters from this run
    best_accuracies.append(best_accuracy)
    best_epochs_list.append(best_epochs)
    best_batch_sizes.append(best_batch_size)

# Calculate averages
avg_best_accuracy = np.mean(best_accuracies)
avg_best_epochs = np.mean(best_epochs_list)
avg_best_batch_size = np.mean(best_batch_sizes)

# Print the average results
print(f"Average Best Accuracy: {avg_best_accuracy:.4f}")
print(f"Average Best Number of Epochs: {avg_best_epochs:.2f}")
print(f"Average Best Batch Size: {avg_best_batch_size:.2f}")

**Note**: The code below creates the results2.csv file in append mode and creates a CSV writer object. It then writes a new row to the CSV file containing the values of epoch, batch_size, test_loss, and test_accuracy.

In [ ]:
# Append this run's result to a local CSV
with open('results2.csv', 'a', newline='') as file:
    csv.writer(file).writerow([epoch, batch_size, test_loss, test_accuracy])

In [ ]:
# To confirm the file was indeed created with content
if os.path.exists('results2.csv'):
    print("File 'results2.csv' exists and was created.")
else:
    print("File 'results2.csv' does not exist or wasn't created.")

In [ ]:
# Check if the file exists before attempting to open it
if os.path.exists('results2.csv'):
    os.system('start "" "results2.csv"')  # This works on Windows to open the file
    # For other operating systems, you might need different commands:
    # os.system('open results2.csv')  # macOS
    # os.system('xdg-open results2.csv')  # Linux
else:
    print("The file 'results2.csv' does not exist.")

In [ ]:
print(os.getcwd())